In [ ]:
!pip install seleniumbase

In [ ]:
!wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get update
!apt-get install -y ./google-chrome-stable_current_amd64.deb
!rm google-chrome-stable_current_amd64.deb

--2026-07-27 20:36:16--  https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
Resolving dl.google.com (dl.google.com)... 209.85.200.91, 209.85.200.93, 209.85.200.190, ...
Connecting to dl.google.com (dl.google.com)|209.85.200.91|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 133561688 (127M) [application/x-debian-package]
Saving to: ‘google-chrome-stable_current_amd64.deb’

google-chrome-stabl 100%[===================>] 127.37M   202MB/s    in 0.6s    

2026-07-27 20:36:17 (202 MB/s) - ‘google-chrome-stable_current_amd64.deb’ saved [133561688/133561688]

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:3 https://dl.google.com/linux/chrome-stable/deb stable InRelease [2,548 B]
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit

In [ ]:
from seleniumbase import SB
from bs4 import BeautifulSoup

def scrape_letterboxd(url, sb):
    #print("Iniciando Chrome con perfil guardado...")

    #with SB(uc=True, xvfb=True, user_data_dir="perfil_chrome_letterboxd") as sb:

    print("Intentando acceso rápido...")
    # uc_open es casi instantáneo. No hace la pausa de 8 segundos.
    sb.uc_open(url)

    estado = sb.execute_script("return document.readyState;")

    # Revisamos si Cloudflare nos bloqueó (si la lista NO está en la página)
    if estado != "complete":
        print("Cookie caducada o Cloudflare detectado. Aplicando evasión profunda (tomará unos 10s)...")
        sb.uc_open_with_reconnect(url, reconnect_time=8)
        sb.uc_gui_click_captcha()

    try:
        # Llegados a este punto, por vía rápida o lenta, ya deberíamos estar dentro
        #sb.wait_for_element(wait_element, timeout=15)
        sb.wait_for_ready_state_complete(timeout=10)
        print("¡Éxito! Estamos dentro de la página real.")

        html = sb.get_page_source()
        return html

    except Exception as e:
        print("\n❌ Error final. Cloudflare no nos dejó pasar esta vez.")
        print("Detalle:", e)

# Ejecutamos la función (¡No necesitas await porque SeleniumBase es síncrono!)
with SB(uc=True, xvfb=True, user_data_dir="perfil_chrome_letterboxd") as sb:
  html =scrape_letterboxd("https://letterboxd.com/film/anora/reviews/by/activity/page/1", sb)

Intentando acceso rápido...
¡Éxito! Estamos dentro de la página real.


In [ ]:
from curl_cffi import requests
from bs4 import BeautifulSoup
import openai
import time

def get_film_ids(url, n_pages=10):
  # Fetch the webpage content
  film_slugs = []
  headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/90.0.4430.212 Safari/537.36"
  }
  for i in range(n_pages):
    url_page = f"{url}/page/{i+1}/"
    response = requests.get(url_page, headers=headers)


    if response.status_code != 200:
      print("Failed to retrieve the web page.")
      exit()

    # Parse the webpage content with BeautifulSoup
    soup = BeautifulSoup(response.content, 'html.parser')

    movies = soup.find_all('div', class_='react-component')
    for poster_div in movies:
        film_slug = poster_div.get('data-item-slug')
        if film_slug:
            film_slugs.append(film_slug)

  return film_slugs

import random

def extract_complete_review(url, sb):
  time.sleep(0.5)
  # headers = {
  #   "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/90.0.4430.212 Safari/537.36"
  # }
  # response = requests.get(url, headers=headers)

  html = scrape_letterboxd(url,sb=sb)

  soup = BeautifulSoup(html, "html.parser")
  print(soup)
  review = soup.find('p')

  return review.text

import re

def get_reviews(film_id, users, n_reviews=20, min_len_review=200, max_len_review=10000, sortedby="entry-rating"):
    data_reviews = []
    print("Iniciando Chrome con perfil guardado...")

    with SB(uc=True, xvfb=True, user_data_dir="perfil_chrome_letterboxd") as sb:
      for k in range(30):
          url = f"https://letterboxd.com/film/{film_id}/reviews/by/{sortedby}/page/{k+1}/"
          print(f"Navegando a la página {k+1}...")

          try:

              # Extraemos el HTML completo

              html = scrape_letterboxd(url, sb=sb)

              soup = BeautifulSoup(html, 'html.parser')

              reviews = soup.find_all('div', class_='listitem')

              # Lista para guardar las reviews extraídas
              print(len(reviews))
              for review in reviews:

                  # Extraer nombre del usuario
                  user_name = review.find('strong', class_='displayname')

                  if user_name is None: continue
                  user_name = user_name.text
                  print(user_name)

                  if user_name in users: continue

                  # Encuentra el elemento <a> con la clase 'context'
                  #url_review = review.find('a', class_='context')['href']
                  url_review = review.find('div', class_='-prose')['data-full-text-url']
                  #print(url_review)

                  # Extraer la fecha de la review
                  date = review.find('span', class_='date').text

                  # Extraer el rating (número de estrellas)

                  rating_elem = review.find('span', class_=re.compile(r'rating|rated'))

                  # Si encuentra el elemento, extrae el texto de las estrellas; si no, devuelve None
                  rating = rating_elem.text.strip() if rating_elem else None

                  # Extraer el texto de la review
                  rev = review.find('div', class_='js-collapsible-text')
                  review_text = '\n'.join([p.get_text() for p in rev.find_all('p')])
                  #print(review_text)

                  if len(review_text) < min_len_review: continue

                  if rev.find('div', class_='collapsed-text') is not None:
                    review_text  = extract_complete_review("https://letterboxd.com/"+url_review, sb)

                  #paragraphs = process_text(review_text)

                  # Añadir la review extraída a la lista de reviews
                  data_reviews.append({
                      'user_name': user_name,
                      'date': date,
                      'rating': rating,
                      'review_text': review_text[:max_len_review]
                      #'paragraphs': paragraphs
                  })

                  users.add(user_name)
                  if len(users) > n_reviews: break

              print("reviews count:", len(users))
              if len(users) > n_reviews: break

          except Exception as e:
              # Si pasaron los 20 segundos y no apareció la lista de reseñas, algo falló.
              print(f"❌ Error en la página {k+1}. ¿Bloqueo persistente?: {e}")
              break


    # Imprimir las reviews extraídas
    text=""
    for rev in data_reviews:
        text+=f"{rev['review_text']}\n\n"

    print(text)

    return data_reviews

def load_info_movie(users= set(), film_id=None, url=None, n_reviews=10, min_len_review=200, max_len_review=1000, review_sort="activity"):
  if url is not None:
    response = requests.get(url)
    soup = BeautifulSoup(response.content, "html.parser")
    film_id = soup.find("meta", property="og:url")["content"].split('/')[-2]
    print(film_id)
  else:
    url = f"https://letterboxd.com/film/{film_id}/"
    response = requests.get(url)
    soup = BeautifulSoup(response.content, "html.parser")


  info=dict()
  try:
    info["title"] = soup.find("meta", property="og:title")["content"]
  except TypeError:
    print(soup)
    raise TypeError

  info["description"] = ""
  if soup.find("meta", property="og:description") is not None:
    info["description"] = soup.find("meta", property="og:description")["content"]

  info["director"] = soup.find("meta", attrs={"name": "twitter:data1"})["content"]
  if soup.find("meta", attrs={"name": "twitter:data2"}) is not None:
    info["average_rating"] = soup.find("meta", attrs={"name": "twitter:data2"})["content"]
  else:
      info["average_rating"] = 0.0
  info["image_url"] = soup.find("meta", property="og:image")["content"]

  general_info="# Película\n\nTítulo:"+ info["title"]
  general_info+="\nDirector:"+ info["director"]
  general_info+="\nDescripción:"+ info["description"]
  general_info+="\nComments:\n"
  #text+="Promedio de calificación:", average_rating)
  #text+="URL de la imagen:", image_url)

  reviews = get_reviews(film_id, users, n_reviews, min_len_review=min_len_review,sortedby=review_sort)
  return info, general_info, reviews


In [ ]:
#films = get_film_ids(f"https://letterboxd.com/dave/list/official-top-250-narrative-feature-films",n_pages=5)
films = get_film_ids(f"https://letterboxd.com/griffinzane/list/criticstop10com-2025", n_pages=5)
films

['one-battle-after-another',
 'sinners-2025',
 'marty-supreme',
 'it-was-just-an-accident',
 'sentimental-value-2025',
 'the-secret-agent-2025',
 'weapons-2025',
 'train-dreams',
 'hamnet',
 'sorry-baby-2025',
 'the-mastermind-2025',
 'if-i-had-legs-id-kick-you',
 'no-other-choice-2025',
 'sirat-2025',
 'blue-moon-2025',
 'frankenstein-2025',
 'bugonia',
 'black-bag-2025',
 'eddington',
 'the-shrouds',
 '28-years-later',
 'wake-up-dead-man',
 'afternoons-of-solitude',
 'misericordia-2024',
 'eephus',
 'caught-by-the-tides',
 'superman-2025',
 'nouvelle-vague-2025',
 'resurrection-2025',
 'the-testament-of-ann-lee',
 'peter-hujars-day',
 'f1',
 'the-phoenician-scheme',
 'sound-of-falling',
 'kpop-demon-hunters',
 'cloud-2024',
 'on-becoming-a-guinea-fowl',
 'the-naked-gun',
 'jay-kelly',
 'die-my-love',
 'twinless',
 'my-undesirable-friends-part-i-last-air-in',
 'grand-tour-2024',
 'the-voice-of-hind-rajab',
 'the-perfect-neighbour',
 'april-2024-1',
 'the-life-of-chuck',
 'highest-2-lo

In [ ]:
import pandas as pd

users = set()

# Extraer los datos (reviews1 es ahora una lista estructurada)
info, general_info, reviews1 = load_info_movie(
    users=users,
    film_id="anora",
    n_reviews=10,
    min_len_review=250,
    review_sort="activity"
)

# Convertir la lista directamente en un DataFrame de Pandas
reviewsDataFrame = pd.DataFrame(reviews1)

# Exportar la tabla a un archivo CSV en el disco
reviewsDataFrame.to_csv("reseñas_anora.csv", index=False, encoding="utf-8")

# Verificar integridad de archivos imprimiendo las 5 primeras entradas
print("¡Archivo 'reseñas_anora.csv' creado con éxito! Aquí tienes una vista previa:")
display(reviewsDataFrame.head())

Iniciando Chrome con perfil guardado...
Navegando a la página 1...
Intentando acceso rápido...
¡Éxito! Estamos dentro de la página real.
19
ava adore
kauan.
samara
Intentando acceso rápido...
¡Éxito! Estamos dentro de la página real.
<html><head></head><body><p>it’s really sad to me that a character driven piece about a woman fails to make a real character out of her. we do not see anora’s desires or emotions outside of a punchline or even outside of her husband who she met the week before. in the film, the distinction between anora’s work and personal lives is blurred to the point where it’s nonexistent. i believe this is where the majority of my problems with the film arise. the only thing she really wants in this film is to find her husband and stay with him. the way it’s presented were led to believe that she actually loves him. of course, as a person i’d say she probably just fell in love with the chance at a better life but as an audience member that’s not driven home. the way th

,user_name,date,rating,review_text
0,samara,19 Oct 2024,★,it’s really sad to me that a character driven ...
1,Drew Burnett Gregory,02 Oct 2024,★★★,Once Igor appears on screen the movie shifts t...
2,clem,23 Sep 2024,★★★★½,the things i would do to get on this bitch fin...
3,Sethsreviews,15 Oct 2024,★★★★½,Appreciated this in ways I didn't expect at al...
4,Paul,08 Nov 2024,★★½,Anora is fun. I’ll say that. After all the hyp...


In [ ]:
from pathlib import Path
from datetime import datetime

# Tomamos las primeras 10 películas de la lista 'films' de la Celda 37
lista_peliculas = films[:10]

lista_resenas = []

for indice, film_slug in enumerate(lista_peliculas):
    print(f"\n--- [Película {indice + 1}/{len(lista_peliculas)}]: Procesando '{film_slug}' ---")
    usuarios_vistos = set()

    try:
        info_meta, _, resenas_pelicula = load_info_movie(
            users=usuarios_vistos,
            film_id=film_slug,
            n_reviews=10,
            min_len_review=250,
            review_sort="activity"
        )

        # Inyectar el contexto a cada reseña para que Pandas sepa de qué película es
        for resena in resenas_pelicula:
            resena["film_id"] = film_slug
            resena["film_title"] = info_meta.get("title", film_slug)
            resena["director"] = info_meta.get("director", "Desconocido")
            lista_resenas.append(resena)

        print(f"✅ Éxito con '{film_slug}': {len(resenas_pelicula)} reseñas capturadas.")
        time.sleep(2) # Pausa de cortesía para estabilizar el socket local de ChromeDriver

    except Exception as error:
        print(f"❌ Fallo al procesar '{film_slug}': {error}")
        continue

# Convertir toda la lista a un DataFrame y guardar en disco
# Nombre de "df_base_inicial" se mantiene puesto que en el futuro este
# script se va a generalizar y no únicamente retornarán el df inicial.
df_base_inicial = pd.DataFrame(lista_resenas)

curr_dir = Path.cwd()
raw_dir = curr_dir / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

# Extraer hora actual
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# Crear nuevo nombre
new_filename = f"src_{timestamp}.csv"
output_path = raw_dir/ new_filename


df_base_inicial.to_csv(output_path, index=False, encoding="utf-8")

print(f"\nExtracción completada. Se guardaron {len(df_base_inicial)} reseñas en '{new_filename}'\n")
display(df_base_inicial.head())


--- [Película 1/10]: Procesando 'one-battle-after-another' ---
Iniciando Chrome con perfil guardado...
Navegando a la página 1...
Intentando acceso rápido...
¡Éxito! Estamos dentro de la página real.
19
eddyburback
mikko
ConnorEatsPants
𝐉
Jake
Jason Concepcion
Preet
fran hoepfner
stavvybaby2
James (Schaffrillas)
zoë rose bryant
Patrick Willems
reviews count: 2
Navegando a la página 2...
Intentando acceso rápido...
¡Éxito! Estamos dentro de la página real.
19
mary
Iman Vellani
itscharlibb
Intentando acceso rápido...
¡Éxito! Estamos dentro de la página real.
<html><head></head><body><p>feel like i haven’t watched a film in aaaaages coz of being on honeymoon and being in paris and having a weird body clock at the moment where im just sort of up all hours of the day working but i finally went to see the pta in paris with finn after being in the studio. it was literally the hottest theatre ive ever been to which just sort of added to the stress but yeah this was super cool and everyone cam